In [1]:
from ultralytics import YOLO
from pathlib import Path
import csv

# ============================================================
# 1. LOAD MODEL
# ============================================================

MODEL_PATH = r"D:\Hydroponics AI\models\combined_baseline.pt"

model = YOLO(MODEL_PATH)

print("Model loaded successfully!")
print("Classes:")
print(model.names)


# ============================================================
# 2. INPUT / OUTPUT FOLDERS
# ============================================================

TEST_FOLDER = Path(r"D:\Hydroponics AI\unseen_test")

OUTPUT_FOLDER = Path(r"D:\Hydroponics AI\unseen_predictions_combined_baseline")
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)


# ============================================================
# 3. FIND ALL IMAGES
# ============================================================

image_extensions = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}

images = [
    p for p in TEST_FOLDER.rglob("*")
    if p.is_file()
    and p.suffix.lower() in image_extensions
]

print("\n" + "=" * 70)
print("UNSEEN DATASET")
print("=" * 70)

print(f"Found {len(images)} images.")


# ============================================================
# 4. RUN INFERENCE
# ============================================================

results_csv = OUTPUT_FOLDER / "results.csv"

rows = []

for i, image_path in enumerate(images, start=1):

    print("\n" + "=" * 60)
    print(f"[{i}/{len(images)}] Testing: {image_path.name}")

    # --------------------------------------------------------
    # Relative location of image
    # --------------------------------------------------------

    relative_path = image_path.relative_to(TEST_FOLDER)

    # Folder immediately containing image
    condition = image_path.parent.name

    # --------------------------------------------------------
    # YOLO inference
    # --------------------------------------------------------

    results = model.predict(
        source=str(image_path),
        imgsz=640,
        conf=0.25,
        save=True,
        save_txt=True,
        project=str(OUTPUT_FOLDER),
        name="predictions",
        exist_ok=True,
        verbose=False
    )

    result = results[0]


    # ========================================================
    # NO DETECTION
    # ========================================================

    if result.boxes is None or len(result.boxes) == 0:

        print("Prediction: No Detection")

        rows.append({
            "image": image_path.name,
            "relative_path": str(relative_path),
            "condition": condition,
            "predicted_class": "No Detection",
            "confidence": 0.0
        })

        continue


    # ========================================================
    # FIND HIGHEST-CONFIDENCE DETECTION
    # ========================================================

    best_confidence = -1
    best_class_id = None

    for box in result.boxes:

        class_id = int(box.cls[0])
        confidence = float(box.conf[0])

        if confidence > best_confidence:

            best_confidence = confidence
            best_class_id = class_id


    # ========================================================
    # GET CLASS NAME
    # ========================================================

    predicted_class = model.names[best_class_id]

    print(f"Prediction : {predicted_class}")
    print(f"Confidence : {best_confidence:.4f}")


    # ========================================================
    # SAVE ONE ROW PER IMAGE
    # ========================================================

    rows.append({
        "image": image_path.name,
        "relative_path": str(relative_path),
        "condition": condition,
        "predicted_class": predicted_class,
        "confidence": round(best_confidence, 4)
    })


# ============================================================
# 5. SAVE CSV
# ============================================================

with open(
    results_csv,
    "w",
    newline="",
    encoding="utf-8"
) as file:

    writer = csv.DictWriter(
        file,
        fieldnames=[
            "image",
            "relative_path",
            "condition",
            "predicted_class",
            "confidence"
        ]
    )

    writer.writeheader()
    writer.writerows(rows)


# ============================================================
# 6. SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("BASELINE TESTING COMPLETED")
print("=" * 70)

print(f"Total images tested : {len(images)}")
print(f"CSV rows generated  : {len(rows)}")

print("\nPredictions saved to:")
print(OUTPUT_FOLDER)

print("\nResults CSV:")
print(results_csv)

Model loaded successfully!
Classes:
{0: 'Calcium Deficiency', 1: 'Healthy', 2: 'Magnesium Deficiency', 3: 'Nitrogen Deficiency', 4: 'Phosphorus Deficiency', 5: 'Potassium Deficiency'}

UNSEEN DATASET
Found 434 images.

[1/434] Testing: 0603192346_jpg.rf.34b01f05ec1f21bab08e051961d3a04a.jpg
Results saved to D:\Hydroponics AI\unseen_predictions_combined_baseline\predictions
1 label saved to D:\Hydroponics AI\unseen_predictions_combined_baseline\predictions\labels
Prediction : Potassium Deficiency
Confidence : 0.9671

[2/434] Testing: b2f91e1d-f5c0-4dde-b9b7-fbe4da02a11f.png
Results saved to D:\Hydroponics AI\unseen_predictions_combined_baseline\predictions
2 labels saved to D:\Hydroponics AI\unseen_predictions_combined_baseline\predictions\labels
Prediction : Potassium Deficiency
Confidence : 0.3176

[3/434] Testing: calcium.jpg
Results saved to D:\Hydroponics AI\unseen_predictions_combined_baseline\predictions
3 labels saved to D:\Hydroponics AI\unseen_predictions_combined_baseline\pred